In [0]:
%sql
-- =============================================================================
-- ETL Silver: Ingesta limpia, tipada y con Media Recortada (P05 a P95)
-- Granularidad destino: 1 fila por (video_id, hashtag)
-- Elimina el sesgo de outliers extremos en reproducciones (colas 5% y 95%)
-- =============================================================================
WITH bronze_typed AS (
    SELECT
        search_type,
        search_hashtag,
        video_id,
        NULLIF(description, 'None')           AS description,
        to_timestamp(created_at)              AS created_at,
        video_url,
        NULLIF(region, 'None')                AS region,
        CAST(duration AS INT)                 AS duration,
        CAST(video_width AS INT)              AS video_width,
        CAST(video_height AS INT)             AS video_height,
        ratio,
        CAST(is_ad AS BOOLEAN)                AS is_ad,
        CAST(is_photo AS BOOLEAN)             AS is_photo,
        CAST(is_paid_content AS BOOLEAN)      AS is_paid_content,
        description_language,
        CAST(plays AS BIGINT)                 AS plays,
        CAST(likes AS BIGINT)                 AS likes,
        CAST(comments AS BIGINT)              AS comments,
        CAST(shares AS BIGINT)                AS shares,
        CAST(saves AS BIGINT)                 AS saves,
        author_id,
        author_unique_id,
        author_nickname,
        CAST(author_verified AS BOOLEAN)      AS author_verified,
        NULLIF(author_signature, 'None')      AS author_signature,
        hashtags,
        fecha_ingesta
    FROM tiktok_data_eng.bronze.tiktok_bronze
),
umbrales_recorte AS (
    -- Cálculo de percentiles P05 y P95 a nivel de video único para eliminar colas sesgadas
    SELECT 
        percentile_approx(plays, 0.05) AS p05_plays,
        percentile_approx(plays, 0.95) AS p95_plays
    FROM (
        SELECT DISTINCT video_id, plays 
        FROM bronze_typed
        WHERE plays IS NOT NULL
    )
),
silver_filtrado AS (
    SELECT
        b.search_type,
        b.search_hashtag,
        b.video_id,
        b.description,
        b.created_at,
        b.video_url,
        b.region,
        b.duration,
        b.video_width,
        b.video_height,
        b.ratio,
        b.is_ad,
        b.is_photo,
        b.is_paid_content,
        b.description_language,
        b.plays,
        b.likes,
        b.comments,
        b.shares,
        b.saves,
        b.author_id,
        b.author_unique_id,
        b.author_nickname,
        b.author_verified,
        b.author_signature,
        hashtags_explodido                    AS hashtag,
        b.fecha_ingesta
    FROM bronze_typed b
    CROSS JOIN umbrales_recorte u
    LATERAL VIEW EXPLODE(
        from_json(regexp_replace(b.hashtags, "'", '"'), 'array<string>')
    ) AS hashtags_explodido
    WHERE b.plays BETWEEN u.p05_plays AND u.p95_plays
      AND hashtags_explodido IS NOT NULL
      AND hashtags_explodido != 'None'
      AND hashtags_explodido != ''
      -- Regla de Negocio: Relevancia Temática exclusiva en Ingeniería de Datos
      AND b.search_hashtag IN ('ingenieriadedatos', 'databricks', 'dataengineering', 'pyspark', 'dataengineer')
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY b.video_id, hashtags_explodido 
        ORDER BY b.fecha_ingesta DESC
    ) = 1
)
MERGE INTO tiktok_data_eng.silver.silver_tiktok AS target
USING silver_filtrado AS source
    ON target.video_id = source.video_id
   AND target.hashtag  = source.hashtag
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;



In [0]:
%sql
-- Verificación de calidad post-carga y métricas de media recortada
SELECT 
    count(*) as total_filas,
    count(distinct video_id) as videos_unicos,
    count(distinct hashtag) as hashtags_unicos,
    round(avg(plays), 2) as media_recortada_plays,
    round(avg(likes), 2) as media_recortada_likes,
    count(*) - count(distinct concat(video_id, '__', hashtag)) as duplicados_pk
FROM tiktok_data_eng.silver.silver_tiktok;